<a href="https://colab.research.google.com/github/Text-Machine/mask-predict/blob/main/chr-paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

# CHR Paper notebook

This notebook aims to reproduce all figures and tables that inform the analysis of the CHR paper.

In [ ]:
!git clone https://github.com/Text-Machine/mask-predict.git

In [ ]:
%cd mask-predict

In [ ]:
!pip install -q -e .

In [ ]:
import pandas as pd
import json
from tqdm import tqdm
from explain import *
from pathlib import Path
from collections import Counter
import seaborn as sns

In [ ]:
%cd /content

In [ ]:
!gdown 125wfZ1P9MfFZ19XS2SCfallsnoNhLCUf

In [ ]:
!unzip -o "chr-data.zip"

note about explanation of data and choices to made here

In [ ]:
collection,genre_suffix = 'blb',''
if collection == 'blb':
  genre_suffix = '_with_genre'

TargetMaskedToken = 'machine' # the token to be masked in the target sentence
try:
  import google.colab
  originalFolder = '.' # change to '.' when working in colab
  dataPath = '.' # change to '.' when working in colab 
  processedFolder = '.' # change '.' when working in colab
except:
  originalFolder = 'masking_data' # change to '.' when working in colab
  dataPath = 'input_data' # change to '.' when working in colab 
  processedFolder = 'gradient_data' # change '.' when working in colab

predCol = "pred_bert_1760_1900"
resultType = 'pred_kw_filtered' # pred | pred_kw_filter

print(f"This analysis focuses on '{TargetMaskedToken}'.")

We are loading the data frame with deduplicated sentences.

In [ ]:
df_sent_all = pd.read_csv(f'{dataPath}/{collection}_{TargetMaskedToken}_clusters{genre_suffix}_deduplicated.tsv', index_col=0, sep='\t').reset_index(drop=True)
print(f'We have {df_sent_all.shape[0]} unique sentences for the target token {TargetMaskedToken} in the {collection} collection.')


In [ ]:
# load the original sentences with the predicted tokens
df_sent = pd.read_csv(f'{dataPath}/{collection}_{TargetMaskedToken}{genre_suffix}_{resultType}.tsv', index_col=0, sep='\t').reset_index(drop=True)
print(f'We have {df_sent.shape[0]} sentences that produced human predictions for the target token {TargetMaskedToken} in the {collection} collection.')
df_ig = pd.read_csv(f'{processedFolder}/results_{collection}_{TargetMaskedToken}_{resultType}_processed.csv', index_col=0 )
print(f'We have {df_ig.shape[0]} explanations for the target token {TargetMaskedToken} in the {collection} collection.')


In [ ]:
df_ig['id'].max()

In [ ]:
df_ig.columns

In [ ]:
ids = df_ig[df_ig["Target"]=='child'].sort_values(by='Target_score', ascending=False)['id'].unique()[:10]

In [ ]:
df_ig[df_ig['id'].isin(ids)][['Token', 'Score', 'Target']].to_csv('test.csv')

In [ ]:
df_sent['id'] = range(df_sent.shape[0])

In [ ]:
from typing import List, Tuple, Literal

def merge_subtokens(
    tokens: List[Tuple[str, float]],
    agg: Literal["mean", "max", "min", "sum", "first"] = "mean"
) -> List[Tuple[str, float]]:
    """
    Merge WordPiece-style subtokens (prefixed with '##') back into their
    parent token, aggregating the associated scores.

    Args:
        tokens: List of (token, score) tuples, e.g. from a tokenizer/model
                output such as [("Hello", 0.9), ("##world", 0.8), ...]
        agg: How to combine scores for merged tokens - "mean", "max",
             "min", "sum", or "first" (keep only the first subtoken's score).

    Returns:
        List of (merged_token, aggregated_score) tuples.
    """
    if not tokens:
        return []

    agg_funcs = {
        "mean": lambda scores: sum(scores) / len(scores),
        "max": max,
        "min": min,
        "sum": sum,
        "first": lambda scores: scores[0],
    }
    if agg not in agg_funcs:
        raise ValueError(f"Unknown agg method: {agg!r}")
    combine = agg_funcs[agg]

    result = []
    current_text_parts = [tokens[0][0]]
    current_scores = [tokens[0][1]]

    for token, score in tokens[1:]:
        if token.startswith("##"):
            current_text_parts.append(token[2:])
            current_scores.append(score)
        else:
            # flush the previous merged token
            result.append(("".join(current_text_parts), combine(current_scores)))
            current_text_parts = [token]
            current_scores = [score]

    # flush the last one
    result.append(("".join(current_text_parts), combine(current_scores)))

    return result

In [ ]:
df_sent_all['token_pseudo_ppl_processed'] = df_sent_all['token_pseudo_ppl'].apply(lambda x: merge_subtokens(eval(x)))


In [ ]:
df_sent['token_pseudo_ppl'] = df_sent['token_pseudo_ppl'].apply(lambda x: eval(x))
df_sent['token_pseudo_ppl_processed'] = df_sent['token_pseudo_ppl'].apply(merge_subtokens)
df_sent_perpl = df_sent[['id','genre','token_pseudo_ppl_processed']].explode('token_pseudo_ppl_processed', ignore_index=True)

df_ig[df_ig.Target.isin(['machine', 'machines'])].shape,df_sent_perpl.shape

In [ ]:
df_sent_perpl[['Token','perplexity']] = pd.DataFrame(df_sent_perpl.token_pseudo_ppl_processed.tolist(), index=df_sent_perpl.index)

In [ ]:
from collections import defaultdict, Counter
def index_duplicates(tokens):
    indexed_sent = []
    wf = Counter(tokens)
    counts = defaultdict(int)
    for t in tokens:
        if wf[t] >= 2:
            counts[t]+=1
            indexed_sent.append(f'{t}_{counts[t]}')
        else:
            indexed_sent.append(t)
    return indexed_sent


In [ ]:
#df_ig['Token_id'] = df_ig.groupby('id').apply(lambda x: x.groupby(['Target'])['Token'].transform(index_duplicates))
df_ig['Token_id'] = df_ig.apply(lambda x: x['Token'] if x['Token'] != '[MASK]' else x['Target'], axis=1)




In [ ]:
df_ig['Token_id'] = df_ig.groupby(['id','Target'])['Token_id'].transform(index_duplicates)
df_sent_perpl['Token_id'] = df_sent_perpl.groupby(['id'])['Token'].transform(index_duplicates)


In [ ]:
df_ig_perpl = df_ig.merge(df_sent_perpl, on=['id','Token_id'], how='left')
df_ig_perpl.shape, df_ig.shape

# Percentage of sentences with human predictions

In [ ]:
round((df_sent.shape[0] / df_sent_all.shape[0]) * 100 ,2)

Here we load the words that we selected as human prediction, i.e. these are words predicted by BLERT possible referring to human fillers for the masked machine token.

In [ ]:
# with open(f'{dataPath}/100_freq_pred_BL_edit.txt') as f:
#     human_words = f.read().splitlines()
df_words = pd.read_csv(f"{dataPath}/word_human_clusters_v5_edit.csv")
human_words = list(df_words[df_words['human_likelihood'] > .1]['word'])

print(human_words[:10], len(human_words))

We create a new column where filter the prediction, only retaining the human words.

In [ ]:
for colName in ['pred_bert_contemporary', 'pred_bert_1760_1900']:
    df_sent[f'{colName}_human'] = df_sent[colName].apply(
        lambda x: {w:s for w, s in dict(eval(x)).items() if w in human_words})


We look at results by decade, therefore adding decade column to the data frames

In [ ]:

if collection == 'hmd':
    df_sent_all['date'] = df_sent_all['year']
    df_sent['date'] = df_sent['year']
df_sent_all['decade'] = df_sent_all['date'].apply(lambda x: int(x/10)*10)
df_sent['decade'] = df_sent['date'].apply(lambda x: int(x/10)*10)

## Analysis: Perplexity

In [ ]:
import numpy as np
perpl_upper = 500
df_sent_all['perplexity'] = df_sent_all['token_pseudo_ppl_processed'].apply(lambda x: np.mean([min(v,perpl_upper) for k, v in x]))
df_sent_all['log_perplexity'] = np.log(df_sent_all['perplexity'])

In [ ]:
df_sent_all.groupby('genre')['perplexity'].mean().plot(kind='bar', title='Average Perplexity by Genre', ylabel='Average Perplexity', xlabel='Genre')

In [ ]:
df_sent_all['human_prediction'] = df_sent_all['pred_bert_1760_1900'].apply(lambda x: bool({w:s for w, s in dict(eval(x)).items() if w in human_words}))
df_sent_all['human_prediction_max'] = df_sent_all['pred_bert_1760_1900'].apply(lambda x: max([s for w, s in dict(eval(x)).items() if w in human_words]+[0]))

In [ ]:
df_sent_all.groupby('human_prediction')['perplexity'].mean().plot(kind='bar', title='Average Perplexity by Human Prediction', ylabel='Average Perplexity', xlabel='Human Prediction')

In [ ]:
df_sent_all.groupby(['genre','human_prediction'])['perplexity'].mean().unstack().plot(kind='bar', title='Average Perplexity by Genre and Human Prediction', ylabel='Average Perplexity', xlabel='Genre')

In [ ]:
df_sent_all[df_sent_all['decade'].between(1800,1899)].groupby(['decade','human_prediction'])['perplexity'].median().unstack().plot( title='Average Perplexity by Genre and Human Prediction', ylabel='Average Perplexity', xlabel='Genre')

In [ ]:
sns.barplot(y='perplexity',x='decade', data=df_sent_all[df_sent_all['decade'].between(1800,1899)], hue='human_prediction')

In [ ]:
threshold = .4
df_sent_all[f'human_prediction_max_{threshold}'] = df_sent_all['human_prediction_max'] > threshold
df_sent_all.groupby(['genre',f'human_prediction_max_{threshold}'])['perplexity'].mean().unstack().plot(kind='bar', title='Average Perplexity by Genre and Human Prediction', ylabel='Average Perplexity', xlabel='Genre')

In [ ]:
df_sent_all.groupby(['decade',f'human_prediction_max_{threshold}'])['perplexity'].mean().unstack().plot(kind='bar', title='Average Perplexity by Genre and Human Prediction', ylabel='Average Perplexity', xlabel='Genre')

In [ ]:
df_sent.columns

### Analysis: Distribution of predictions

BLERT is not always equally convinced of it's predictions. Below we plot the distribution of it confidence scores for the human words it predicted instead of machine. These are the probabilities that a word fits the given context in the position of the masked token. We see that 

In [ ]:
import itertools
scores_human = list(itertools.chain.from_iterable(df_sent['pred_bert_1760_1900_human'].apply(lambda x: list(x.values()))))
scores_all = list(itertools.chain.from_iterable(df_sent['pred_bert_1760_1900'].apply(lambda x: list(dict(eval(x)).values()))))

print("Distribution of the human prediction scores for the target token '{TargetMaskedToken}' in the {collection} collection.")
#pd.Series(scores_human).plot(kind='density')
pd.Series(scores_human).hist(bins=100, alpha=0.9, label='human predictions')
#pd.Series(scores_all).plot(kind='density', alpha=0.5)


### Analysis: timeline for all predictions

Below we plot the number of unique sentences containing 'human' predictions for masked machine tokens. We plot by decade, for all the human predictions, higher than a set confidence threshold. 

While maybe not drastically, we observe a small rise in the relative number of 'atypical' sentences or 'atypical' language use.

In [ ]:
time_unit = 'decade' # change to 'date' for yearly analysis
thresholds = [0.1, 0.2, 0.5]
for threshold in thresholds:
    df_sent_filtered = df_sent[df_sent['pred_bert_1760_1900_human'].apply(lambda x: max(list(x.values())+[.0]) > threshold)]
    (df_sent_filtered.groupby(time_unit).size()/df_sent_all.groupby(time_unit).size()).loc[1800:1899].plot(alpha=0.9,
                                                                                                            title=f'', 
                                                                                                            xlabel=f'{time_unit}',
                                                                                                            figsize=(15, 4))


In [ ]:
import seaborn as sns
for colName in [ 'pred_bert_1760_1900']:
    df_sent_all[f'{colName}_human'] = df_sent_all[colName].apply(
        lambda x: {w:s for w, s in dict(eval(x)).items() if w in human_words})

In [ ]:
from matplotlib import rcParams

# figure size in inches
rcParams['figure.figsize'] = 5.27,5.27
thresholds = [0.1, 0.25, 0.5]
for threshold in thresholds:
    df_sent_all[f'max_human_value_{threshold}'] = df_sent_all['pred_bert_1760_1900_human'].apply(lambda x: max(list(x.values())+[.0]) >= threshold)
    sns.lineplot(y=f'max_human_value_{threshold}',x='decade',data=df_sent_all[df_sent_all['date'].between(1800,1899)], alpha=0.9)

In [ ]:
threshold = 0.1

df_sent_all[f'max_human_value_{threshold}'] = df_sent_all['pred_bert_1760_1900_human'].apply(lambda x: max(list(x.values())+[.0]) >= threshold)
sns.lineplot(y=f'max_human_value_{threshold}',x='decade',
             data=df_sent_all[df_sent_all['date'].between(1800,1899)],
             alpha=0.9, hue='genre', hue_order=['Fiction','Non-fiction'], palette=['#1f77b4', '#ff7f0e'], legend='full')

# Analysis for semantic clusters

In [ ]:
df_words = pd.read_csv(f"{dataPath}/word_human_clusters_v5_edit.csv")

df_words.fillna('', inplace=True)
threshold = 0.1

In [ ]:
cols = list(df_words.columns[2:])
for c in cols:
    exec(f"{c}2word = {{}}")

    for i, row in df_words.iterrows():
        if row['human_likelihood'] <= threshold:
            continue

        for hc in row[c].split(';'):
            exec(f"if hc not in {c}2word and hc != '': {c}2word[hc] = []")
            if hc != '':
                exec(f"{c}2word[hc].append(row['word'])")
    print(f"Created a dictionary for {c} with {len(eval(c+'2word'))} keys.")


In [ ]:
df_ig_target = df_ig[['id','Target']].drop_duplicates(inplace=False)
df_ig_target = df_ig_target[df_ig_target.Target.isin(human_words)]
df_ig_target = df_ig_target.merge(df_words, left_on='Target', right_on='word', how='left')
 

In [ ]:
visualise_col = 'human_subcategory'
df_ig_target[f'{visualise_col}_split'] = df_ig_target[visualise_col].apply(lambda x: x.split(';'))

In [ ]:
from explain.visualise import plot_semantic_squares
df_ig_target_exp = df_ig_target.explode(f'{visualise_col}_split').reset_index(drop=True)
fig, ax = plot_semantic_squares(df_ig_target_exp, column=f'{visualise_col}_split')

In [ ]:
for k,v in chr_categories2word.items():
    print(f"{k}: {', '.join(v[:5])}")

In [ ]:
df_ig_target_exp[f'{visualise_col}_split'].value_counts(normalize=True).plot(kind='bar', figsize=(15, 4), title=f'Counts of the top 10 {visualise_col} for the target token {TargetMaskedToken} in the {collection} collection.')

In [ ]:
len(chr_categories2word.keys())

In [ ]:
time_unit = 'decade' # change to 'date' for yearly analysis
thresholds = [ 0.1, 0.2, 0.5] # compare the results for different thresholds to see how the results change based on the confidence in the predictions
wordList = semantic_clusters2word['body_part']#hc2word['occupation_domestic_service'] # change these words to analyze a different theme
print(f"Analyzing the theme of '{wordList}' for the target token '{TargetMaskedToken}' in the {collection} collection.")
for threshold in thresholds:
    df_sent_filtered = df_sent[df_sent[f'{colName}_human'].apply(
        lambda x: max(list({w:s for w,s in x.items() if w in wordList}.values())+[.0]) > threshold
                )
            ]
    (df_sent_filtered.groupby(time_unit).size() / df_sent_all.groupby(time_unit).size()).loc[1800:1899].plot(alpha=0.8, 
                                                                                                             title=f'',
                                                                                                               xlabel=f'{time_unit}',
                                                                                                               figsize=(15, 4))


In [ ]:
from matplotlib import rcParams
import matplotlib.pyplot as plt

# figure size in inches
rcParams['figure.figsize'] = 4.0,4.0
fig, axes = plt.subplots(
    nrows=4,
    ncols=2,
    figsize=(12, 16),
    sharex=True,      # optional
    sharey=False      # optional
)

thresholds = [0.1, 0.25, 0.5]
categories = list(chr_categories2word.keys())
axes = axes.flatten()

for i, ax in enumerate(axes):
    wordList = chr_categories2word[categories[i]]#hc2word['occupation_domestic_service'] # change these words to analyze a different theme
    for threshold in thresholds:
        df_sent_all[f'max_human_value_{threshold}'] = df_sent_all['pred_bert_1760_1900_human'].apply(
            lambda x: max(list({w:s for w,s in x.items() if w in wordList}.values())+[.0]) > threshold
            )
        sns.lineplot(y=f'max_human_value_{threshold}',x='decade',ax=ax, legend=False,
                 data=df_sent_all[df_sent_all['date'].between(1800,1899)], alpha=0.9,
                )
    
    ax.set_title(f"Group {categories[i]}")
    #ax.legend().remove() 

plt.tight_layout()
plt.show()

In [ ]:
# figure size in inches
rcParams['figure.figsize'] = 4.0,4.0
fig, axes = plt.subplots(
    nrows=4,
    ncols=2,
    figsize=(12, 16),
    sharex=True,      # optional
    sharey=False      # optional
)

threshold = 0.1
categories = list(chr_categories2word.keys())
axes = axes.flatten()

for i, ax in enumerate(axes):
    wordList = chr_categories2word[categories[i]]#hc2word['occupation_domestic_service'] # change these words to analyze a different theme
    
    df_sent_all[f'max_human_value_{threshold}'] = df_sent_all['pred_bert_1760_1900_human'].apply(
            lambda x: max(list({w:s for w,s in x.items() if w in wordList}.values())+[.0]) > threshold
            )
    sns.lineplot(y=f'max_human_value_{threshold}',x='decade',ax=ax, legend=False,
                 hue='genre', hue_order=['Fiction','Non-fiction'],
                 data=df_sent_all[df_sent_all['date'].between(1800,1899)], alpha=0.9,
                )
    
    ax.set_title(f"Group {categories[i]}")
    #ax.legend().remove() 

plt.tight_layout()
plt.show()

### Analysis: timeline for a selected thema

By changing the `wordList` variable below, we can plot timelines for specific subset or theme of predictions. 

The children theme shows a pronounced upward trend. This could be one of the subquestions we address in the paper: how and why the increasing confusion of child and machine?

In [ ]:
time_unit = 'decade' # change to 'date' for yearly analysis
thresholds = [ 0.1, 0.2, 0.5] # compare the results for different thresholds to see how the results change based on the confidence in the predictions
wordList = ['child','children', 'boy','boys','girl','girls']  # change these words to analyze a different theme
for threshold in thresholds:
    df_sent_filtered = df_sent[df_sent[f'{colName}_human'].apply(
        lambda x: max(list({w:s for w,s in x.items() if w in wordList}.values())+[.0]) > threshold
                )
            ]
    (df_sent_filtered.groupby(time_unit).size() / df_sent_all.groupby(time_unit).size()).loc[1800:1899].plot(alpha=0.8, 
                                                                                                             title=f'',
                                                                                                               xlabel=f'{time_unit}',
                                                                                                               figsize=(15, 4))


### Analysis: results by genre

The results hold when splitting the data by genre. Upward trend appears in both fiction and non-fiction but, more articulate in the form}er.

In [ ]:

time_unit = 'decade' # change to 'date' for yearly analysis
threshold = 0.1 # use higher thresholds for more conservative analysis, i.e. only sentences with a high probability of the tokens in wordList being predicted are included in the analysis
wordList = ['child','children', 'boy','boys','girl','girls'] # change these words to analyze a different theme

df_sent_filtered = df_sent[df_sent['pred_bert_1760_1900_human'].apply(
    lambda x: max(list({w:s for w,s in x.items() if w in wordList}.values())+[.0]) > threshold
    )]
plot_df = df_sent_filtered.groupby([time_unit, 'genre']).size() / df_sent_all.groupby([time_unit, 'genre']).size()
plot_df_rel = (df_sent_filtered.groupby([time_unit, 'genre']).size() / df_sent_all.groupby([time_unit, 'genre']).size()).reset_index()
sns.lineplot(data=plot_df_rel, x=time_unit, y=0, hue='genre', marker='o')

In [ ]:
from matplotlib import rcParams

# figure size in inches
rcParams['figure.figsize'] = 15.0,4.0
threshold = 0.1
#wordList = semantic_clusters2word['body_part']#hc2word['occupation_domestic_service'] # change these words to analyze a different theme
wordList = ['child','children', 'boy','boys','girl','girls'] # change these words to analyze a different theme
df_sent_all[f'max_human_value_{threshold}'] = df_sent_all['pred_bert_1760_1900_human'].apply(
        lambda x: max(list({w:s for w,s in x.items() if w in wordList}.values())+[.0]) > threshold
        )
sns.lineplot(y=f'max_human_value_{threshold}',x='decade',
                 data=df_sent_all[df_sent_all['date'].between(1800,1899)], 
                 hue='genre',hue_order=['Fiction', 'Non-fiction'],alpha=0.9)

### Analysis: Distribution of prediction

Which human words does BLERT predict instead of ''machine''? And why? 

We refined the analysis below by allowing to refine the threshold, i.e. to focus only on "confident" predictions etc.

In [ ]:

thresholds = [0.01,0.1, 0.25, 0.5 ]# change this threshold to see how the results change based on the confidence in the predictions


In [ ]:
dfs_temp = []
for threshold in thresholds:

    df_sent_filtered = df_sent[df_sent[f'{colName}_human'].apply(lambda x: max(list(x.values())+[0]) > threshold)]
    preds = Counter([w for l in df_sent_filtered[f'{colName}'].values for w,s in eval(l) if (w in human_words) & (s > threshold) ])
    df_threshold = pd.DataFrame([(w, c/sum(preds.values())) for w, c in preds.most_common(100)]
             ).rename(columns={0: 'Predicted token', 1: f'threshold={threshold}'}
                      ).set_index('Predicted token'
                                  )#.plot(kind='bar', title=f'Top 100 predicted tokens for the target token {TargetMaskedToken}.', 
                                   #      ylabel='Count', xlabel='Predicted token', figsize=(15,5))
    dfs_temp.append(df_threshold)

In [ ]:
prob_by_token = pd.concat(dfs_temp, axis=1)
prob_index =prob_by_token.mean(axis=1).sort_values(ascending=False).index

prob_by_token.loc[prob_index[:25]].plot(kind='bar', title=f'Top 25 predicted tokens for the target token {TargetMaskedToken}.', 
                             ylabel='Probability', xlabel='Predicted token',width=0.8, figsize=(15,5))


In [ ]:
list(chr_categories2word.keys())

In [ ]:
dfs_temp = []
wordList = []
wordList = chr_categories2word['age_status']
print(wordList)
for threshold in thresholds:

    df_sent_filtered = df_sent[df_sent[f'{colName}_human'].apply(
        lambda x: max(list({w:s for w,s in x.items() if w in wordList}.values())+[.0]) > threshold
        )]
    preds = Counter([w for l in df_sent_filtered[f'{colName}'].values for w,s in eval(l) if (w in wordList) & (s > threshold) ])
    df_threshold = pd.DataFrame([(w, c/sum(preds.values())) for w, c in preds.most_common(100)]
             ).rename(columns={0: 'Predicted token', 1: f'threshold={threshold}'}
                      ).set_index('Predicted token'
                                  )#.plot(kind='bar', title=f'Top 100 predicted tokens for the target token {TargetMaskedToken}.', 
                                   #      ylabel='Count', xlabel='Predicted token', figsize=(15,5))
    dfs_temp.append(df_threshold)

In [ ]:
prob_by_token = pd.concat(dfs_temp, axis=1)
prob_index =prob_by_token.mean(axis=1).sort_values(ascending=False).index

prob_by_token.loc[prob_index[:25]].plot(kind='bar', title=f'Top 25 predicted tokens for the target token {TargetMaskedToken}.', 
                             ylabel='Probability', xlabel='Predicted token',width=0.8, figsize=(15,5))


### Analysis: explainability

Which words drive the predictions, and especially, how to interpet this increasing "confusion" or children with machines? What does it tell us about changes in the discourse about machines.

What we compute is not just how words contribute to the prediction, this tells us often more about what type of sentences are included in our data. We look at which words explain the difference between machine and human words, i.e. which words drive the prediction towards 'human' and away from 'machine'.

More technically, for each word in the sentence we compute pairwise differences between ''machine'' and ''human'' predictions (for the masked token) for each token in the sentence.

In [ ]:
# Create row order within each id and Target
df_ig_perpl["row_idx"] = df_ig.groupby(["id", "Target"]).cumcount()

# Extract machine/machines scores
machine_scores = (
    df_ig_perpl[df_ig_perpl["Target"].isin(["machine", "machines"])]
    [["id", "row_idx", "Score_normalized"]]
    .rename(columns={"Score_normalized": "machine_score"})
)

# Match each row with the corresponding machine row
df_ig_perpl = df_ig_perpl.merge(
    machine_scores,
    on=["id", "row_idx"],
    how="left"
)

# Subtract
df_ig_perpl["diff"] = df_ig_perpl["Score_normalized"] - df_ig_perpl["machine_score"]

# Optional: remove helper column
df_ig_perpl.drop(columns=["row_idx", "machine_score"], inplace=True)

In [ ]:
df_ig_perpl[(df_ig_perpl['id'] == 11)].Target.unique()

In [ ]:
df_ig_perpl[(df_ig_perpl['id'] == 11) & (df_ig_perpl['Target'].isin(['machine', 'machines']))][['Target', 'Score_normalized', 'diff']].head(3)

In [ ]:
df_ig_perpl[(df_ig_perpl['id'] == 11) & (df_ig_perpl['Target'].isin(['girl']))][['Target', 'Score_normalized', 'diff']].head(3)

## Analysis: Explanation scores across all human word predictions

Below is a general analysis, highlighting which context words explain the drifting aways of explanations from the machine.

The threshold, again, regulates, the confidence.

The `df_result` dataframe shows the words that exhibit the highest difference between machine and human words. Below we provide tools for analysing these words, and the sentences, in which they appear, in more depth.

In [ ]:
df_ig_perpl.rename({'Token_x':'Token'}, inplace=True, axis=1)

In [ ]:
threshold = 0.25

targetTokens = ['machine','machines'] # we look at the predictions for all the non machine words
df_comparisonConcept = df_ig_perpl[
    (~df_ig_perpl['Target'].isin(targetTokens)) & # we exclude the target token itself, as we are interested in other tokens that are predictive of the contrastive concept
    (df_ig_perpl['Target_score'].between(threshold, 1.0))
                ].groupby('Token').agg(
                        count=('id', 'count'),identifiers=('id', set),avg_diff=('diff', 'mean'), avg_score=('Score_normalized', 'mean')
                    ).reset_index()


In [ ]:
# please note that this repeats sentences, this acros all sentences with all the filtered keywords
min_count = 5
df_result = df_comparisonConcept[df_comparisonConcept['count'] >= min_count].sort_values(by='avg_diff', ascending=False)
df_result.head(10)

## Analysis: explainabiliy and perplexity

Q: Is the perplexity significantly higher for these context words?

In [ ]:
topn = 20
comp_tokens = df_result.Token.to_list()[:min([topn,df_result.shape[0]])]
all_tokens = df_result.Token.to_list()

In [ ]:
len(all_tokens)

In [ ]:
perpl_threshold = 500
sum(df_ig_perpl[df_ig_perpl.Target.isin(['machine','machines'])].perplexity > perpl_threshold) / len(df_ig_perpl[df_ig_perpl.Target.isin(['machine','machines'])])

In [ ]:

df_ig_perpl['perplexity_threshold'] = df_ig_perpl['perplexity']

df_ig_perpl.loc[df_ig_perpl['perplexity'] > perpl_threshold,'perplexity_threshold'] = perpl_threshold

In [ ]:
df_ig_perpl[df_ig_perpl.Token.isin(all_tokens)].fillna(.0)[['Score_normalized','perplexity_threshold']].corr()

In [ ]:
df_ig_perpl['perpl_token'] = False
df_ig_perpl.loc[df_ig_perpl.Token.isin(comp_tokens),'perpl_token'] = True

df_ig_perpl[(df_ig_perpl.Target.isin(['machine','machines'])) & \
            df_ig_perpl.Token.isin(all_tokens)
            ].groupby('perpl_token')['perplexity_threshold'].mean().plot(kind='bar')

## Analysis: Explanation scores for a specific theme


The code below, repeats the contextual analyis, but focussing on specific theme, defined by the `comparisonTokens` variable. Here we highlight which context words explain the drifting aways of explanations from the machine, with regard to a specific theme.

The `threshold`  regulates, the confidence.

`comarisonTokens` defines a set of words which we compare against machine predictions.

The `df_result` dataframe shows the words that exhibit the highest difference between machine and human words. Below we provide tools for analysing these words, and the sentences, in which they appear, in more depth.

In [ ]:
list(chr_categories2word.keys())

In [ ]:
#comparisonTokens = ['child','children', 'boy','boys','girl','girls']
comparisonTokens = chr_categories2word['age_status']
df_comparisonConcept = df_ig[
    (df_ig['Target'].isin(comparisonTokens)) & \
    (df_ig['Target_score'].between(0.4, 1.0)) # we exclude the target token itself, as we are interested in other tokens that are predictive of the contrastive concept
                ].groupby('Token').agg(
                        count=('id', 'count'),identifiers=('id', set),avg_diff=('diff', 'mean'), avg_score=('Score_normalized', 'mean')
                    ).reset_index()


In [ ]:
# please note that this repeats sentences, this acros all sentences with all the filtered keywords
min_count = 5
df_result = df_comparisonConcept[df_comparisonConcept['count'] >= min_count].sort_values(by='avg_score', ascending=False)
df_result.head(20)

In [ ]:
#df_result.avg_score.plot(kind='hist', bins=100, title=f'Histogram of average scores for tokens predictive of the contrastive concept.', xlabel='Average score', ylabel='Count')

# Analysis: Zooming in on sentences based on context words

You can zoom in a sentences with using the predictive context words. Change the `id` variable below with one of the index numbers of the`df_result` dataframe.

The sentence are ranked, with examples where the context word obtains the highest scores, first.

You can change the following variable:

- `id` selected from the `df_result` dataframe
- `sort_value` how do sort the sentences, i.e. where the impact of the target word is thelargest ('Score_normalized') or makes the biggest difference ('diff')

In [ ]:
pd.set_option('display.max_colwidth', 200)

In [ ]:
id =  380#1637
sort_value = 'diff' #'Score_normalized' | 'diff'


identifiers_ranked = df_ig.loc[df_ig['id'].isin(list(df_result.loc[id].identifiers)) \
                             & (df_ig['Token']==df_result.loc[id].Token) \
                             & df_ig['Target'].isin(comparisonTokens)
                             
                             ].sort_values(by=sort_value, ascending=False)['id'].unique()
sentences_ranked = df_sent.iloc[list(identifiers_ranked)].head(20)
sentences_ranked.currentSentence

In [ ]:
# Optional: save and expeort sentences
name = 'child'
outPath = Path('sentences')
outPath.mkdir(exist_ok=True)
sentences_ranked.to_csv(outPath / f'{name}_sentences.csv', index=False)

In [ ]:
modelName = "Livingwithmachines/bert_1760_1900"
explainer = MaskedLMExplainer(model_name=modelName, device=pick_device())

## Analysis: Inspect IG for one sentence

In [ ]:
idx = 4384
target_token = 'predicted' # 'actual' | 'predicted'

sentence = df_sent.iloc[idx].maskedSentence

if target_token == 'actual':
    target_token = df_sent.iloc[idx].targetExpression
elif target_token == 'predicted':
    target_token =  [w for w, v in sorted(
       df_sent.iloc[idx].pred_bert_contemporary_human.items(), key=lambda x: x[1], reverse=True)
                     if w in wordList][0]
print(target_token)

In [ ]:

highlight_context_tokens(explainer, sentence, target=target_token, word_agg="mean")

In [ ]:

highlight_context_tokens(explainer, sentence, target=target_token, word_agg="mean")

## Inspect sentences based on Context and Predicted token

In [ ]:
predictedToken = 'men'
contextToken = 'animal'
sort_value = 'diff' #'Score_normalized' | 'diff'

ids = df_ig[(df_ig['Token'].str.lower() == contextToken) & (df_ig['Target'].str.lower() == predictedToken)
                    ].sort_values(sort_value,ascending=False).id.values

In [ ]:
idx = 4495
wordList = human_words
sentence = df_sent.iloc[idx].maskedSentence
targets = list(set(df_ig[df_ig['id'] == idx].Target.unique()).intersection(set(wordList)))
target_expression = df_sent.iloc[idx].targetExpression

In [ ]:
print(f"Sentence: {sentence}")
print(f"Targets: {targets}")
print(f"Targets: {target_expression}")

In [ ]:
#target =targets[0]

In [ ]:

highlight_context_tokens(explainer, sentence, target=predictedToken, word_agg="mean")

# Notebook still under construction here, please ignore or run at your own risk! :-)

## Genre and Patterns

In [ ]:
df_ig_merged = df_ig.merge(df_sent[['decade','date','genre']], left_on='id',right_index=True, how='left')
df_ig.shape, df_ig_merged.shape

In [ ]:
df_ig_merged.columns

In [ ]:
cutoff = df_ig_merged[df_ig_merged.Score_normalized > 0].Score_normalized.mean()
df_ig_merged_cut = df_ig_merged[df_ig_merged.Score_normalized > cutoff].reset_index(drop=True)
df_ig_merged_cut = df_ig_merged_cut[~df_ig_merged_cut['Token'].isin(['[MASK]','.',',', '!', '?'])]

In [ ]:
grouped = df_ig_merged_cut.groupby('id')


for n in range(2,6):

    df_ig_merged_cut[f'ngram_{n}'] = df_ig_merged_cut['Token']
    for i in range(1, n):
        df_ig_merged_cut[f'ngram_{n}'] = df_ig_merged_cut[f'ngram_{n}'] + ' ' + grouped['Token'].shift(-i)


    df_ig_merged_cut[f'ngram_mean_{n}'] = (
        df_ig_merged_cut.groupby('id')['Score_normalized']
        .rolling(window=n)
        .mean()
        .reset_index(level=0, drop=True)
    )
    df_ig_merged_cut[f'ngram_mean_{n}'] = df_ig_merged_cut.groupby('id')[f'ngram_mean_{n}'].shift(-(n-1))

In [ ]:
#comparisonTokens = ['child','children', 'boy','boys','girl','girls']
n = 4
comparisonTokens = chr_categories2word['age_status']
target_tokens = ['machine','machines'] # we look at the predictions for all the non machine words
df_comparisonConcept = df_ig_merged_cut[
    #(df_ig_merged_cut['Target'].isin(comparisonTokens)) & \
    (~df_ig_merged_cut['Target'].isin(target_tokens)) & \
    (df_ig_merged_cut['Target_score'].between(0.4, 1.0)) # we exclude the target token itself, as we are interested in other tokens that are predictive of the contrastive concept
                ].groupby(f'ngram_{n}').agg(
                        count=('id', 'count'),identifiers=('id', set),avg_diff=('diff', 'mean'), avg_score=('Score_normalized', 'mean')
                    ).reset_index()


In [ ]:
# please note that this repeats sentences, this acros all sentences with all the filtered keywords
min_count = 2
df_result = df_comparisonConcept[df_comparisonConcept['count'] >= min_count].sort_values(by='avg_score', ascending=False)
df_result.head(20)

# Genre

In [ ]:
#comparisonTokens = ['child','children', 'boy','boys','girl','girls']
comparisonTokens = chr_categories2word['age_status']
df_comparisonConcept = df_ig_merged[
    (df_ig_merged['Target'].isin(comparisonTokens)) & \
    
    (df_ig_merged['Target_score'].between(0.4, 1.0)) # we exclude the target token itself, as we are interested in other tokens that are predictive of the contrastive concept
                ].groupby(['genre','Token']).agg(
                        count=('id', 'count'),identifiers=('id', set),avg_diff=('diff', 'mean'), avg_score=('Score_normalized', 'mean')
                    ).reset_index()


In [ ]:

genre_scores = (
    df_comparisonConcept
    .pivot(index='Token', columns='genre', values=['avg_score', 'avg_diff', 'count'])
)

df_genre_diff = (
    genre_scores[('avg_score', 'Fiction')] - genre_scores[('avg_score', 'Non-fiction')]
).fillna(0).sort_values(ascending=False)

df_genre_diff.tail(20)

In [ ]:
# please note that this repeats sentences, this acros all sentences with all the filtered keywords
min_count = 5
df_result = df_comparisonConcept[df_comparisonConcept['count'] >= min_count].sort_values(by='avg_score', ascending=False)
df_result.head(20)

## Historical Experiments

In [ ]:
min_sentence_length = 20
max_sentence_length = 200   
min_confidence = .1
column_name = 'Score_normalized' # Score_normalized | diff

df_ig_merged[(df_ig_merged['Target'].isin(['machine','machines'])) & \
             (df_ig_merged['Target_score'].between(min_confidence,1.0)) & \
             (df_ig_merged['Sentence_length'].between(min_sentence_length, max_sentence_length))
             ].groupby(['decade','id'])[column_name].mean().groupby('decade').mean().loc[1800:1890].plot(title=f'Average {column_name} in scores for tokens predictive of the contrastive concept over time.', xlabel='Decade', ylabel='Average difference in scores', figsize=(15, 4))

df_ig_merged[(~df_ig_merged['Target'].isin(['machine','machines'])) & \
             (df_ig_merged['Target_score'].between(min_confidence,1.0)) & \
             (df_ig_merged['Sentence_length'].between(min_sentence_length, max_sentence_length))
             ].groupby(['decade','id'])[column_name].mean().groupby('decade').mean().loc[1800:1890].plot()

df_ig_merged[(df_ig_merged['Target'].isin(comparisonTokens)) & \
             (df_ig_merged['Target_score'].between(min_confidence,1.0)) & \
             (df_ig_merged['Sentence_length'].between(min_sentence_length, max_sentence_length))
             ].groupby(['decade','id'])[column_name].mean().groupby('decade').mean().loc[1800:1890].plot()

In [ ]:
df_ig['sent_diff'] = df_ig.groupby('id')['diff'].transform(lambda x: x.mean())
df_ig['sent_diff'] = df_ig.groupby('id')['diff'].transform(lambda x: x.mean())
sent_diff_scores = df_ig[(~df_ig['Target'].isin(['machine','machines'])) & \
                          (df_ig['Target_score'].between(.1,1.0)) & \
                          (df_ig['Sentence_length'] > 20)
                          ].drop_duplicates(subset=['id'])['sent_diff']
#sent_diff_scores.plot(kind='hist', bins=100, title=f'Histogram of average difference in scores for tokens predictive of the contrastive concept.', xlabel='Average difference in scores', ylabel='Count')

In [ ]:
sent_diff_scores = df_ig[~df_ig['Target'].isin(['machine','machines'])].drop_duplicates(subset=['id'])['sent_diff']
sent_diff_scores.plot(kind='hist', bins=100, title=f'Histogram of average difference in scores for tokens predictive of the contrastive concept.', xlabel='Average difference in scores', ylabel='Count')

## Analysis: Sentences where we observe largest changes 

In [ ]:
min_confidence = .1
max_confidence = 1.0
min_sentence_length = 50
max_sentence_length = 200
n_examples = 10000

most_changes = df_ig[(~df_ig['Target'].isin(['machine','machines'])) & \
                      (df_ig['Target_score'].between(min_confidence, max_confidence)) & \
                      (df_ig['Sentence_length'].between(min_sentence_length, max_sentence_length))
                      ].sort_values(by='sent_diff', ascending=False)[['id','sent_diff','Target','Token']]['id'].unique()[:n_examples]

In [ ]:
pd.set_option('display.max_colwidth', 200)
df_sel_sent = df_sent.iloc[most_changes]
df_sel_sent.currentSentence

In [ ]:
min_confidence = .1
max_confidence = 1.0
min_sentence_length = 50
max_sentence_length = 200
n_examples = 100

minimal_changes = df_ig[(~df_ig['Target'].isin(['machine','machines'])) & \
                      (df_ig['Target_score'].between(min_confidence, max_confidence)) & \
                      (df_ig['Sentence_length'].between(min_sentence_length, max_sentence_length)) & \
                      (df_ig['sent_diff'].between(-0.005,0.005))
                      ].sort_values(by='sent_diff', ascending=True)[['id','sent_diff','Target','Token']]['id'].unique()[:n_examples]

In [ ]:
pd.set_option('display.max_colwidth', 200)
df_sel_sent = df_sent.iloc[minimal_changes]
df_sel_sent.currentSentence

In [ ]:
idx = 17071 #15972
target_token = 'predicted' # 'actual' | 'predicted'

sentence = df_sent.iloc[idx].maskedSentence

if target_token == 'actual':
    target_token = df_sent.iloc[idx].targetExpression
elif target_token == 'predicted':
    target_token =  [w for w, v in sorted(
       df_sent.iloc[idx].pred_bert_contemporary_human.items(), key=lambda x: x[1], reverse=True)
                     if w in wordList][0]
print(target_token)

In [ ]:

highlight_context_tokens(explainer, sentence, target=target_token, word_agg="mean")

## qa: which predicted words change the context dependencies the least?

## Linear trends

In [ ]:
df_ig_by_token = df_ig_merged.groupby(['Token','decade'])['Score_normalized'].mean().reset_index()

In [ ]:
words = [w for w,v in Counter(df_ig['Token']).items() if v > 100]

In [ ]:
from sklearn.linear_model import LinearRegression
def get_linear_regression_slope(df, x_col='decade', y_col='Score_normalized'):
    X = df[[x_col]].values
    y = df[y_col].values
    model = LinearRegression()
    model.fit(X, y)
    return model.coef_[0] # Return the slope

In [ ]:
slopes = df_ig_by_token.groupby('Token').apply(lambda x: get_linear_regression_slope(x))
slopes.loc[words].sort_values(ascending=False)

In [ ]:
token = 'sewing'
df_ig_merged[df_ig_merged.Token==token].groupby('decade')['Score_normalized'].mean().plot(title=f'Average score for the token "{token}" over time.', xlabel='Decade', ylabel='Average score', figsize=(15, 4))

# Topic Modelling

In [ ]:
from explain.topicbert_viz import embed_sentences

In [ ]:
df_topic_model = df_sent.copy()

In [ ]:
sentences = df_sent.currentSentence.tolist()

embeddings = embed_sentences(
    sentences,
    checkpoint=modelName,
    batch_size=32,
    normalize=True,
    device=pick_device()
)

print(embeddings.shape)
# torch.Size([3, 768])

In [ ]:
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
import plotly.express as px

In [ ]:
embeddings = np.asarray(embeddings)
df_topic_model["embeddings"] = list(embeddings)

In [ ]:
 tsne = TSNE(
        n_components=2,
        perplexity=min(30, len(df_topic_model) - 1),
        random_state=42,
        init="pca",
    )

In [ ]:
coords = tsne.fit_transform(embeddings)

df_topic_model["tsne_x"] = coords[:, 0]
df_topic_model["tsne_y"] = coords[:, 1]


In [ ]:
df_topic_model = df_topic_model.merge(
    df_ig[['id', 'sent_diff']].drop_duplicates(),
    left_index=True, right_on='id', how='left'
    )
df_topic_model.shape

In [ ]:
df_topic_model['sent_diff_abs'] = df_topic_model['sent_diff'].abs().fillna(0)

In [ ]:
df_topic_model['sent_diff_abs']

In [ ]:
fig = px.scatter(
        df_topic_model,
        x="tsne_x",
        y="tsne_y",
        color='decade',
        size='sent_diff_abs',
        hover_data=["currentSentence", "decade"],
        title="t-SNE of Sentence Embeddings",
        

    )

fig.update_traces(marker=dict( opacity=0.7), selector=dict(mode='markers'))
fig.update_layout(
        xaxis_title="t-SNE 1",
        yaxis_title="t-SNE 2",
        template="plotly_white",
    )

fig.show()

In [ ]:
# from explain.topicbert_viz import draw_scatterplot_regions

# fig_with_regions = draw_scatterplot_regions(
#     fig,
#     df,
#     region_col="decade",
#     label_col="decade",
# )

# fig_with_regions.show()


In [ ]:
# import plotly.express as px
# from explain.topicbert_viz import draw_clustered_scatterplot_regions

# fig_clustered_regions = px.scatter(
#     df,
#     x="tsne_x",
#     y="tsne_y",
#     color="decade",
#     hover_data=["currentSentence", "decade"],
#     title="Scatterplot regions from automatic clustering",
# )

# fig_clustered_regions, clustered_regions_df, clustered_regions_summary = draw_clustered_scatterplot_regions(
#     fig_clustered_regions,
#     df,
#     x_col="tsne_x",
#     y_col="tsne_y",
#     method="kmeans",
#     n_clusters=6,
#     min_points=4,
# )

# fig_clustered_regions.show()


In [ ]:
from explain.topicbert_viz import plot_topicbert_topics

In [ ]:
df_topic_model['currentSentence_clean'] = df_topic_model['currentSentence'].str.lower().replace(r'machine|machines', '', regex=True).str.lower().str.strip()

In [ ]:
topicbert_frame, topicbert_summary, topicbert_doc_topic_distributions, topicbert_fig = plot_topicbert_topics(
    model=explainer.model,
    dataframe=df_topic_model,
    text_column="currentSentence_clean",
    tokenizer=explainer.tokenizer,
    max_rows=-1,
    batch_size=8,
    max_length=96,
    embed_device="cpu",
    cluster_method="kmeans",
    n_clusters=20,
)

topicbert_summary

topicbert_doc_topic_distributions.head()

#topicbert_fig.show()

In [ ]:
display(topicbert_frame.currentSentence_clean[:3])
display(df_topic_model.currentSentence_clean[:3])

In [ ]:
df_topic_concat = pd.concat([topicbert_frame, df_topic_model.reset_index()], ignore_index=True, axis=1)

In [ ]:
df_topic_concat.columns = list(topicbert_frame.columns) + list(df_topic_model.reset_index().columns)
df_topic_concat = df_topic_concat.loc[:,~df_topic_concat.columns.duplicated()].copy()

In [ ]:
df_topic_concat.columns

In [ ]:
df_topic_dist_concat = pd.concat([topicbert_doc_topic_distributions, df_topic_model.reset_index()], ignore_index=True, axis=1)
df_topic_dist_concat.columns = list(topicbert_doc_topic_distributions.columns) + list(df_topic_model.reset_index().columns)
df_topic_dist_concat = df_topic_dist_concat.loc[:,~df_topic_dist_concat.columns.duplicated()].copy()
df_topic_dist_concat

In [ ]:
#topicbert_frame_merged = topicbert_frame.merge(df[['currentSentence_clean','currentSentence','decade']], left_on='currentSentence_clean', right_on='currentSentence_clean', how='left')

In [ ]:
fig = px.scatter(
        df_topic_concat,
        x="tsne_x",
        y="tsne_y",
        color='topic_label',
        size='sent_diff_abs',
        hover_data=["currentSentence", "topic_name"],
        title="t-SNE of Sentence Embeddings",
        

    )

fig.update_traces(marker=dict(opacity=0.6), selector=dict(mode='markers'))
fig.update_layout(
        xaxis_title="t-SNE 1",
        yaxis_title="t-SNE 2",
        template="plotly_white",
    )

fig.show()

In [ ]:
path = Path("visualisation")
path.mkdir(exist_ok=True)

fig.write_html(f"{path}/topicbert_visualisation_low_diff.html")

In [ ]:
df_topic_dist_concat['sent_diff_z'] = (df_topic_dist_concat['sent_diff'] - df_topic_dist_concat['sent_diff'].mean()) / df_topic_dist_concat['sent_diff'].std()
df_topic_dist_concat['sent_diff_bool'] = df_topic_dist_concat['sent_diff_z'] >= 1

In [ ]:
topic_label = 15
topiclab =df_topic_dist_concat.groupby(['sent_diff_bool'])[[topic_label for topic_label in df_topic_dist_concat.columns if topic_label.startswith('topic_prob_')]
                                                 ].mean()#.plot(kind='bar', title=f'Average topic probability for sentences with high vs low difference in scores for tokens predictive of the contrastive concept.', xlabel='Sentences with high vs low difference in scores', ylabel='Average topic probability', figsize=(15,5))

In [ ]:
topiclab.T.plot(kind='bar', title=f'Average topic probability for sentences with high vs low difference in scores for tokens predictive of the contrastive concept.', xlabel='Sentences with high vs low difference in scores', ylabel='Average topic probability', figsize=(15,5))

In [ ]:
df_topic_dist_concat['sent_length'] = df_topic_dist_concat['currentSentence'].str.split().apply(len)
df_topic_dist_concat.columns

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from explain import regression_analysis as ra

analysis = ra.RegressionAnalysis(df_topic_dist_concat, y="sent_diff",
                            x=["sent_length"] + [f"topic_prob_{i}" for i in range(20)],
                            y_range="-1_1")
analysis.fit(model="fractional_logit")
print(analysis.summary())


In [ ]:

analysis.plot_actual_vs_predicted()
analysis.plot_residuals()
analysis.plot_coefficients()
analysis.plot_correlation_heatmap()

print(analysis.vif())
print(analysis.performance_metrics())